In [0]:
# Databricks notebook source
import time, traceback
import pyspark.sql.functions as F
from pyspark.sql import Window

# --------------------------
# Config
# --------------------------
g_env = "DEV"
g_ucBronze = "dev_hub_bronze"
g_ucSilver = "dev_hub_silver"
v_p_srcSchema = "lh_ax_idr"

META_TBL = "dev_bronze.poc._meta"
SRC_TS_COL = "data_received_utc_dttm"

# --------------------------
# Helpers
# --------------------------
def norm_tbl(x: str) -> str:
    return (x or "").strip().lower()

def build_table_pk_dict(meta_df):
    """
    Builds: { normalized_table_name: [pk1, pk2, ...] }
    """
    rows = meta_df.select("TABLE_NM", "PK_COL_NM").collect()  # keep; usually small metadata
    d = {}
    for r in rows:
        t = norm_tbl(r["TABLE_NM"])
        c = (r["PK_COL_NM"] or "").strip()
        if not t or not c:
            continue
        d.setdefault(t, []).append(c)
    # de-dupe pk list while preserving order
    for t in list(d.keys()):
        seen = set()
        d[t] = [c for c in d[t] if not (c in seen or seen.add(c))]
    return d

def validate_table(sdf, table: str, pk_cols: list, ts_col: str):
    cols = set(sdf.columns)
    missing = [c for c in ([ts_col] + pk_cols) if c not in cols]
    if missing:
        raise ValueError(f"Missing required columns in {table}: {missing}")

def dedup_and_overwrite(table: str, pk_cols: list):
    """
    Full refresh: read bronze table -> dedup by pk + latest timestamp -> overwrite silver table
    """
    start = time.time()

    src_tbl = f"{g_ucBronze}.{v_p_srcSchema}.{table}"
    tgt_tbl = f"{g_ucSilver}.{v_p_srcSchema}.{table}"

    sdf = spark.table(src_tbl)

    # explicit validation for clearer failures
    validate_table(sdf, table, pk_cols, SRC_TS_COL)

    w = Window.partitionBy(*[F.col(c) for c in pk_cols]).orderBy(F.col(SRC_TS_COL).desc())
    deduped = (
        sdf.withColumn("_rn", F.row_number().over(w))
           .filter(F.col("_rn") == 1)
           .drop("_rn")
    )

    deduped.write.mode("overwrite").saveAsTable(tgt_tbl)

    elapsed = time.time() - start
    print(f"✅ {table} refreshed -> {tgt_tbl} | {elapsed:.2f}s")

# --------------------------
# Load tables + meta
# --------------------------
df_tables = spark.sql(f"""
    SELECT table_name
    FROM {g_ucSilver}.information_schema.tables
    WHERE table_catalog = '{g_ucSilver}'
      AND table_schema  = '{v_p_srcSchema}'
""")

silver_tables = set([norm_tbl(r["table_name"]) for r in df_tables.collect()])

df_meta = spark.table(META_TBL)
table_pk_dict = build_table_pk_dict(df_meta)

# (Optional) Only process tables that exist in silver (prevents "table not found" at target layer)
table_pk_dict = {t: pks for t, pks in table_pk_dict.items() if t in silver_tables}

print(f"Tables to process: {len(table_pk_dict)}")
# --------------------------
# Run sequentially with aggregated errors
# --------------------------
errors = []
for tbl, pk_cols in table_pk_dict.items():
    try:
        print(f"Processing {tbl} | PK={pk_cols}")
        dedup_and_overwrite(tbl, pk_cols)
    except Exception as e:
        errors.append((tbl, repr(e), traceback.format_exc()))
        print(f"❌ {tbl} failed: {e}")

if errors:
    msg = "\n\n".join([f"TABLE={t}\nERR={err}\n{tb}" for t, err, tb in errors])
    raise Exception("Some tables failed:\n" + msg)

In [0]:
import time, traceback
import pyspark.sql.functions as F
from pyspark.sql import Window
from concurrent.futures import ThreadPoolExecutor, as_completed

g_ucBronze = "dev_hub_bronze"
g_ucSilver = "dev_hub_silver"
v_p_srcSchema = "lh_ax_idr"
META_TBL = "dev_bronze.poc._meta"
SRC_TS_COL = "data_received_utc_dttm"
p_maxWorkers = 4  # keep low

def norm_tbl(x: str) -> str:
    return (x or "").strip().lower()

def build_table_pk_dict(meta_df):
    rows = meta_df.select("TABLE_NM", "PK_COL_NM").collect()
    d = {}
    for r in rows:
        t = norm_tbl(r["TABLE_NM"])
        c = (r["PK_COL_NM"] or "").strip()
        if not t or not c:
            continue
        d.setdefault(t, []).append(c)
    for t in list(d.keys()):
        seen = set()
        d[t] = [c for c in d[t] if not (c in seen or seen.add(c))]
    return d

def validate_table(sdf, table: str, pk_cols: list, ts_col: str):
    cols = set(sdf.columns)
    missing = [c for c in ([ts_col] + pk_cols) if c not in cols]
    if missing:
        raise ValueError(f"Missing required columns in {table}: {missing}")

def dedup_and_overwrite(table: str, pk_cols: list):
    start = time.time()
    src_tbl = f"{g_ucBronze}.{v_p_srcSchema}.{table}"
    tgt_tbl = f"{g_ucSilver}.{v_p_srcSchema}.{table}"

    sdf = spark.table(src_tbl)
    validate_table(sdf, table, pk_cols, SRC_TS_COL)

    w = Window.partitionBy(*[F.col(c) for c in pk_cols]).orderBy(F.col(SRC_TS_COL).desc())
    deduped = (
        sdf.withColumn("_rn", F.row_number().over(w))
           .filter(F.col("_rn") == 1)
           .drop("_rn")
    )
    deduped.write.option("mergeSchema", "true").format("delta").mode("overwrite").saveAsTable(tgt_tbl)

    elapsed = time.time() - start
    return f"✅ {table} | {elapsed:.2f}s"

df_tables = spark.sql(f"""
    SELECT table_name
    FROM {g_ucSilver}.information_schema.tables
    WHERE table_catalog = '{g_ucSilver}'
      AND table_schema  = '{v_p_srcSchema}'
""")
silver_tables = set([norm_tbl(r["table_name"]) for r in df_tables.collect()])

df_meta = spark.table(META_TBL)
table_pk_dict = build_table_pk_dict(df_meta)
table_pk_dict = {t: pks for t, pks in table_pk_dict.items() if t in silver_tables}

futures = {}
errors = []
with ThreadPoolExecutor(max_workers=p_maxWorkers) as ex:
    for tbl, pk_cols in table_pk_dict.items():
        futures[ex.submit(dedup_and_overwrite, tbl, pk_cols)] = tbl

for fut in as_completed(futures):
    tbl = futures[fut]
    try:
        print(fut.result())
    except Exception as e:
        errors.append((tbl, repr(e), traceback.format_exc()))
        print(f"❌ {tbl} failed: {e}")

if errors:
    msg = "\n\n".join([f"TABLE={t}\nERR={err}\n{tb}" for t, err, tb in errors])
    raise Exception("Some tables failed:\n" + msg)

In [0]:
import time, traceback
import pyspark.sql.functions as F
from pyspark.sql import Window
from concurrent.futures import ThreadPoolExecutor, as_completed

g_ucBronze = "dev_hub_bronze"
g_ucSilver = "dev_hub_silver"
v_p_srcSchema = "lh_ax_idr"
META_TBL = "dev_bronze.poc._meta"
SRC_TS_COL = "data_received_utc_dttm"
p_maxWorkers = 4  # keep low

def norm_tbl(x: str) -> str:
    return (x or "").strip().lower()

def build_table_pk_dict(meta_df):
    rows = meta_df.select("TABLE_NM", "PK_COL_NM").collect()
    d = {}
    for r in rows:
        t = norm_tbl(r["TABLE_NM"])
        c = (r["PK_COL_NM"] or "").strip()
        if not t or not c:
            continue
        d.setdefault(t, []).append(c)
    for t in list(d.keys()):
        seen = set()
        d[t] = [c for c in d[t] if not (c in seen or seen.add(c))]
    return d

def validate_table(sdf, table: str, pk_cols: list, ts_col: str):
    cols = set(sdf.columns)
    missing = [c for c in ([ts_col] + pk_cols) if c not in cols]
    if missing:
        raise ValueError(f"Missing required columns in {table}: {missing}")

def dedup_and_overwrite(table: str, pk_cols: list):
    start = time.time()
    src_tbl = f"{g_ucBronze}.{v_p_srcSchema}.{table}"
    tgt_tbl = f"{g_ucSilver}.{v_p_srcSchema}.{table}"

    sdf = spark.table(src_tbl)
    validate_table(sdf, table, pk_cols, SRC_TS_COL)

    w = Window.partitionBy(*[F.col(c) for c in pk_cols]).orderBy(F.col(SRC_TS_COL).desc())
    deduped = (
        sdf.withColumn("_rn", F.row_number().over(w))
           .filter(F.col("_rn") == 1)
           .drop("_rn")
    )
    deduped.write.mode("overwrite").saveAsTable(tgt_tbl)

    elapsed = time.time() - start
    return f"✅ {table} | {elapsed:.2f}s"

df_tables = spark.sql(f"""
    SELECT table_name
    FROM {g_ucSilver}.information_schema.tables
    WHERE table_catalog = '{g_ucSilver}'
      AND table_schema  = '{v_p_srcSchema}'
""")
silver_tables = set([norm_tbl(r["table_name"]) for r in df_tables.collect()])

df_meta = spark.table(META_TBL)
table_pk_dict = build_table_pk_dict(df_meta)
table_pk_dict = {t: pks for t, pks in table_pk_dict.items() if t in silver_tables}

futures = {}
errors = []
with ThreadPoolExecutor(max_workers=p_maxWorkers) as ex:
    for tbl, pk_cols in table_pk_dict.items():
        futures[ex.submit(dedup_and_overwrite, tbl, pk_cols)] = tbl

for fut in as_completed(futures):
    tbl = futures[fut]
    try:
        print(fut.result())
    except Exception as e:
        errors.append((tbl, repr(e), traceback.format_exc()))
        print(f"❌ {tbl} failed: {e}")

if errors:
    msg = "\n\n".join([f"TABLE={t}\nERR={err}\n{tb}" for t, err, tb in errors])
    raise Exception("Some tables failed:\n" + msg)

In [0]:
import time, traceback
import pyspark.sql.functions as F
from pyspark.sql import Window
from concurrent.futures import ThreadPoolExecutor, as_completed

# --------------------------
# Config
# --------------------------
g_ucBronze = "dev_hub_bronze"
g_ucSilver = "dev_hub_silver"
v_p_srcSchema = "lh_ax_idr"

META_TBL = "dev_bronze.poc._meta"
TS_COL   = "data_received_utc_dttm"

p_maxWorkers = 4  # keep low for Spark thread-safety

# --------------------------
# Helpers
# --------------------------
def norm_tbl(x: str) -> str:
    return (x or "").strip().lower()

def build_table_pk_dict(meta_df):
    rows = meta_df.select("TABLE_NM", "PK_COL_NM").collect()
    d = {}
    for r in rows:
        t = norm_tbl(r["TABLE_NM"])
        c = (r["PK_COL_NM"] or "").strip()
        if t and c:
            d.setdefault(t, []).append(c)

    # de-dupe pk list preserving order
    for t in list(d.keys()):
        seen = set()
        d[t] = [c for c in d[t] if not (c in seen or seen.add(c))]
    return d

def validate_required_cols(df, table: str, pk_cols: list, ts_col: str):
    cols = set(df.columns)
    missing = [c for c in ([ts_col] + pk_cols) if c not in cols]
    if missing:
        raise ValueError(f"Missing required columns in {table}: {missing}")

def ensure_target_exists(tgt_tbl: str, df):
    if not spark.catalog.tableExists(tgt_tbl):
        df.limit(0).write.format("delta").mode("overwrite").saveAsTable(tgt_tbl)

def merge_latest_by_pk(table: str, pk_cols: list):
    """
    Read bronze -> dedup latest by PK + TS -> MERGE into silver with schema evolution
    """
    start = time.time()

    src_tbl = f"{g_ucBronze}.{v_p_srcSchema}.{table}"
    tgt_tbl = f"{g_ucSilver}.{v_p_srcSchema}.{table}"

    sdf = spark.table(src_tbl)
    validate_required_cols(sdf, table, pk_cols, TS_COL)

    w = Window.partitionBy(*[F.col(c) for c in pk_cols]).orderBy(F.col(TS_COL).desc())
    s_latest = (
        sdf.withColumn("_rn", F.row_number().over(w))
           .filter(F.col("_rn") == 1)
           .drop("_rn")
    )

    # # Enable schema evolution for MERGE
    # spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

    ensure_target_exists(tgt_tbl, s_latest)

    # null-safe join on PKs
    cond = " AND ".join([f"t.`{c}` <=> s.`{c}`" for c in pk_cols])

    # Use a unique temp view per table to avoid clashes across threads
    tmp_view = f"s_latest_{table}"
    s_latest.createOrReplaceTempView(tmp_view)

    spark.sql(f"""
        MERGE INTO {tgt_tbl} AS t
        USING {tmp_view} AS s
        ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    elapsed = time.time() - start
    return f"✅ {table} merged -> {tgt_tbl} | {elapsed:.2f}s"

# --------------------------
# Build processing plan
# --------------------------
df_tables = spark.sql(f"""
    SELECT table_name
    FROM {g_ucSilver}.information_schema.tables
    WHERE table_catalog = '{g_ucSilver}'
      AND table_schema  = '{v_p_srcSchema}'
""")
silver_tables = set(norm_tbl(r["table_name"]) for r in df_tables.collect())

df_meta = spark.table(META_TBL)
table_pk_dict = build_table_pk_dict(df_meta)

# optional: only tables that exist in silver schema
table_pk_dict = {t: pks for t, pks in table_pk_dict.items() if t in silver_tables}

print(f"Tables to process: {len(table_pk_dict)}")

# --------------------------
# Run in parallel + aggregate failures
# --------------------------
futures = {}
errors = []

with ThreadPoolExecutor(max_workers=p_maxWorkers) as ex:
    for tbl, pk_cols in table_pk_dict.items():
        futures[ex.submit(merge_latest_by_pk, tbl, pk_cols)] = tbl

for fut in as_completed(futures):
    tbl = futures[fut]
    try:
        print(fut.result())
    except Exception as e:
        errors.append((tbl, repr(e), traceback.format_exc()))
        print(f"❌ {tbl} failed: {e}")

if errors:
    msg = "\n\n".join([f"TABLE={t}\nERR={err}\n{tb}" for t, err, tb in errors])
    raise Exception("Some tables failed:\n" + msg)